In [1]:
print("hello world")

hello world


In [2]:
import requests
import pandas as pd

API_KEY = "96c6a7e11526aa348124e9e23aa001a5"  # 일단 테스트용

url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/boxoffice/searchDailyBoxOfficeList.json"

params = {
    "key": API_KEY,
    "targetDt": "20240101"
}

res = requests.get(url, params=params)
data = res.json()

df = pd.DataFrame(data['boxOfficeResult']['dailyBoxOfficeList'])

df.head()

,rnum,rank,rankInten,rankOldAndNew,movieCd,movieNm,openDt,salesAmt,salesShare,salesInten,salesChange,salesAcc,audiCnt,audiInten,audiChange,audiAcc,scrnCnt,showCnt
0,1,1,0,OLD,20203702,노량: 죽음의 바다,2023-12-20,2893509165,38.8,-686119312,-19.2,36926118105,290717,-59184,-16.9,3728850,1706,5793
1,2,2,0,OLD,20212866,서울의 봄,2023-11-22,2666429137,35.8,-649456909,-19.6,118016920547,262422,-64054,-19.6,12117201,1614,5444
2,3,3,0,OLD,20236146,신차원! 짱구는 못말려 더 무비 초능력 대결전 ~날아라 수제김밥~,2023-12-22,565417320,7.6,-69888800,-11,6071745009,58239,-6751,-10.4,626508,834,1671
3,4,4,0,OLD,20235735,아쿠아맨과 로스트 킹덤,2023-12-20,507984981,6.8,-156026099,-23.5,7967584753,49368,-14686,-22.9,772354,832,1611
4,5,5,0,OLD,20235596,트롤: 밴드 투게더,2023-12-20,273538754,3.7,-16945147,-5.8,3383181955,28988,-1952,-6.3,365927,660,981


In [3]:
from datetime import datetime, timedelta
import requests
import pandas as pd

API_KEY = "96c6a7e11526aa348124e9e23aa001a5"

url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/boxoffice/searchDailyBoxOfficeList.json"

start = datetime(2015, 1, 1)
end = datetime(2025, 12, 31)

all_data = []

while start <= end:
    date_str = start.strftime("%Y%m%d")

    params = {
        "key": API_KEY,
        "targetDt": date_str
    }

    res = requests.get(url, params=params)
    data = res.json()

    try:
        daily = data['boxOfficeResult']['dailyBoxOfficeList']
        for row in daily:
            row['date'] = date_str  # 날짜 추가
        all_data.extend(daily)
    except:
        pass

    start += timedelta(days=1)

df = pd.DataFrame(all_data)

print("총 행 개수:", df.shape)

총 행 개수: (29990, 19)


In [4]:
df.to_csv("test_kobis.csv", index=False)

import os
size_mb = os.path.getsize("test_kobis.csv") / (1024 * 1024)
print(f"CSV 크기: {size_mb:.2f} MB")

CSV 크기: 3.77 MB


In [1]:
import pandas as pd

df = pd.read_csv("test_kobis.csv")
print(df.shape)
print(df.columns.tolist())
print("고유 movieCd 수:", df["movieCd"].nunique())

(29990, 19)
['rnum', 'rank', 'rankInten', 'rankOldAndNew', 'movieCd', 'movieNm', 'openDt', 'salesAmt', 'salesShare', 'salesInten', 'salesChange', 'salesAcc', 'audiCnt', 'audiInten', 'audiChange', 'audiAcc', 'scrnCnt', 'showCnt', 'date']
고유 movieCd 수: 2180


In [2]:
import requests
import time

API_KEY = "96c6a7e11526aa348124e9e23aa001a5"

movie_url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"

unique_movie_codes = df["movieCd"].dropna().astype(str).unique()
movie_info_list = []

count = 0

for movie_cd in unique_movie_codes:
    params = {
        "key": API_KEY,
        "movieCd": movie_cd
    }

    try:
        res = requests.get(movie_url, params=params, timeout=10)
        res.raise_for_status()
        data = res.json()

        info = data.get("movieInfoResult", {}).get("movieInfo", {})

        genres = ", ".join(
            [g.get("genreNm", "") for g in info.get("genres", []) if g.get("genreNm")]
        )
        directors = ", ".join(
            [d.get("peopleNm", "") for d in info.get("directors", []) if d.get("peopleNm")]
        )
        nations = ", ".join(
            [n.get("nationNm", "") for n in info.get("nations", []) if n.get("nationNm")]
        )
        companies = ", ".join(
            [c.get("companyNm", "") for c in info.get("companys", []) if c.get("companyNm")]
        )

        movie_info_list.append({
            "movieCd": info.get("movieCd"),
            "movieNm_detail": info.get("movieNm"),
            "movieNmEn": info.get("movieNmEn"),
            "showTm": info.get("showTm"),              # 상영시간
            "prdtYear": info.get("prdtYear"),          # 제작연도
            "openDt_detail": info.get("openDt"),
            "typeNm": info.get("typeNm"),
            "prdtStatNm": info.get("prdtStatNm"),
            "genres": genres,
            "directors": directors,
            "nations": nations,
            "companies": companies,
            "repGenreNm": info.get("repGenreNm"),
            "repNationNm": info.get("repNationNm")
        })

    except Exception as e:
        print(f"[오류] movieCd={movie_cd}: {e}")

    count += 1
    if count % 100 == 0:
        print(f"{count}개 상세정보 수집 완료")

    time.sleep(0.1)

movie_df = pd.DataFrame(movie_info_list)

print("영화 상세정보 shape:", movie_df.shape)
print(movie_df.head())

100개 상세정보 수집 완료
200개 상세정보 수집 완료
300개 상세정보 수집 완료
400개 상세정보 수집 완료
500개 상세정보 수집 완료
600개 상세정보 수집 완료
700개 상세정보 수집 완료
800개 상세정보 수집 완료
900개 상세정보 수집 완료
1000개 상세정보 수집 완료
1100개 상세정보 수집 완료
1200개 상세정보 수집 완료
1300개 상세정보 수집 완료
1400개 상세정보 수집 완료
1500개 상세정보 수집 완료
1600개 상세정보 수집 완료
1700개 상세정보 수집 완료
1800개 상세정보 수집 완료
1900개 상세정보 수집 완료
2000개 상세정보 수집 완료
2100개 상세정보 수집 완료
영화 상세정보 shape: (2180, 14)
  movieCd movieNm_detail movieNmEn showTm prdtYear openDt_detail typeNm  \
0    None           None      None   None     None          None   None   
1    None           None      None   None     None          None   None   
2    None           None      None   None     None          None   None   
3    None           None      None   None     None          None   None   
4    None           None      None   None     None          None   None   

  prdtStatNm genres directors nations companies repGenreNm repNationNm  
0       None                                          None        None  
1       None                 

In [ ]:
merged_df = df.merge(movie_df, on="movieCd", how="left")

print("통합 데이터 shape:", merged_df.shape)
print("컬럼 수:", len(merged_df.columns))
merged_df.head()

In [ ]:
merged_df.to_csv("test_kobis_merged.csv", index=False, encoding="utf-8-sig")

import os
size_mb = os.path.getsize("test_kobis_merged.csv") / (1024 * 1024)
print(f"통합 CSV 크기: {size_mb:.2f} MB")